
Ejercicio 1:

¿Cuál es el objetivo de predicción?

El objetivo de la predicción que he realizado es estimar la probabilidad de que un cliente cancele su servicio de telecomunicaiones. Apartir de sus características demográficas, contractuales y de facturación, como antigüedad (tenure), cargos mensuales, tipo de contrato y servicios contratados.

¿Por qué el problema no puede resolverse con regresión lineal?

Porque primeramente la variable objetivo es categórica, no continua. Aquí es como un boolean es 0 o 1. 

¿Qué interpretación práctica tiene estimar una probabilidad en este contexto?

Estimar una probabilidad transforma el problema de “sí o no” en gestión del riesgo. Esto lo que nos permite es rankear clientes, priorizar acciones de retención y ajustar umbrales de decisión según costos de negocio, transformando el modelo en una herramienta de apoyo para decisiones comerciales, no solo en una predicción técnica.


Ejercicio 2:

¿Cuántas clases tiene?

El problema tiene dos clases, ya que la variable objetivo Churn solo puede tomar dos valores: 0 (No cancela) y 1 (Cancela). Esta binarización se realiza explícitamente al convertir Yes/No a 1/0 durante la preparación del target, definiendo así un problema de clasificación binaria.


¿Existe un orden natural entre las clases?

No. Las clases son nominales, no ordinales. “Cancela” y “No cancela” representan estados mutuamente excluyentes sin jerarquía o gradación intermedia. Por tanto, no existe un orden lógico entre ellas, lo que justifica el uso de regresión logística binaria y no modelos ordinales.


Ejercicio 3:

¿Qué tipo de regresión logística corresponde usar?

Corresponde utilizar regresión logística binaria, ya que la variable objetivo (Churn) presenta únicamente dos clases mutuamente excluyentes (0 = permanece, 1 = cancela) y no existe un orden natural entre ellas.

Logística binaria

Modela directamente la probabilidad de pertenecer a una de dos clases mediante la función sigmoide.
Salida: 

𝑃(𝑌=1∣𝑋)

P(Y=1∣X).
Es el enfoque más simple, interpretable y adecuado cuando solo hay dos categorías.

Logística multinomial

Se emplea cuando existen tres o más clases sin orden. Utiliza la función softmax para estimar simultáneamente la probabilidad de cada clase.
Salida: 

𝑃(𝑌=𝑘∣𝑋)

P(Y=k∣X) para múltiples categorías.

One-vs-Rest (OvR)

Estrategia alternativa para problemas multiclase. Entrena varios modelos binarios independientes, cada uno comparando “una clase vs. todas las demás”. Luego se asigna la clase con mayor probabilidad. Es más flexible, pero menos elegante conceptualmente que la multinomial nativa.


# Minitrabajo Regresión Logística - Telco Customer Churn
## Respuestas Ejercicios 3-10

---

## Ejercicio 3 – Análisis Exploratorio de Datos (EDA)

### 3.1 Análisis univariante

**Variables explicativas:**

- **Continuas:** `tenure` (meses con la compañía: 0-72), `MonthlyCharges` (cargo mensual: 18-118€), `TotalCharges` (cargo total acumulado)
- **Categóricas binarias:** `gender`, `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`
- **Categóricas múltiples:** `Contract` (Month-to-month, One year, Two year), `InternetService` (DSL, Fiber optic, No), `PaymentMethod`

**Variable objetivo (Churn):**
- 73.5% No cancela (0), 26.5% cancela (1)
- Existe desbalance moderado de clases (ratio ~3:1)
- Consecuencia: el modelo puede tender a predecir la clase mayoritaria. Se mitiga usando `class_weight="balanced"` en la regresión logística.

### 3.2 Análisis bivariante

**Relaciones con Churn:**
- Clientes con contratos mensuales tienen mayor tasa de churn (~42%) vs. contratos largos (~3-11%)
- Churn aumenta en tenures bajos (<12 meses) y con cargos mensuales altos
- Fiber optic presenta mayor churn que DSL o sin internet

**Multicolinealidad:**
- `TotalCharges` correlaciona fuertemente con `tenure` (0.83) - esperado, ya que TotalCharges = tenure × promedio mensual
- No se detectan otras correlaciones problemáticas (>0.7) entre predictores

---

## Ejercicio 4 – La regresión logística desde la teoría

### 4.1 La función logística

La función sigmoide transforma cualquier valor real en una probabilidad entre 0 y 1:

$$P(Y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + ... + \beta_p X_p)}}$$

Su forma de "S" garantiza que valores extremos del predictor lineal se compriman hacia 0 o 1, evitando predicciones fuera del rango [0,1] que daría una regresión lineal.

### 4.2 Interpretación de los coeficientes

- **Signo:** β > 0 aumenta P(Churn), β < 0 la disminuye
- **Magnitud:** indica fuerza del efecto, pero no linealmente como en regresión lineal
- Se interpretan en términos de **log-odds**, no como cambios directos en probabilidad

### 4.3 Odds y Odds Ratio

**Odds:** razón entre probabilidad de éxito y fracaso: `odds = P/(1-P)`

**Odds Ratio (OR):** `e^β` - multiplicador de las odds por cada unidad de aumento en X

**Ejemplo práctico:** Si β_Contract=Month-to-month = 1.2, entonces OR = e^1.2 ≈ 3.32. Esto significa que tener contrato mensual multiplica las odds de cancelar por 3.32 comparado con contratos largos.

---

## Ejercicio 5 – Preparación de los datos

### 5.1 Tratamiento de valores faltantes

`TotalCharges` contenía valores no numéricos (espacios) que se convirtieron a NaN con `pd.to_numeric(..., errors='coerce')`. Se eliminaron (~11 filas) porque correspondían a clientes con tenure=0 (sin historial, no aportan información predictiva).

### 5.2 Outliers

En regresión logística, los outliers en X afectan menos que en regresión lineal porque la sigmoide satura en extremos. Se conservan porque pueden representar patrones reales (ej: clientes con cargos muy altos). El escalado mediante `StandardScaler` reduce su influencia al normalizar las escalas.

### 5.3 Escalado y codificación

**Escalado:** Se estandarizan variables continuas (`tenure`, `MonthlyCharges`, `TotalCharges`) para que todas contribuyan equitativamente y mejorar la convergencia del optimizador.

**Codificación:** Variables categóricas se transforman con `pd.get_dummies(drop_first=True)` para evitar multicolinealidad perfecta (dummy trap), creando variables binarias.

---

## Ejercicio 6 – Entrenamiento del modelo

### 6.1 Ajuste del modelo

**Variables utilizadas:** Todas excepto `customerID` (identificador sin valor predictivo). Post-encoding: 46 features.

**Método:** `LogisticRegression` con:
- `solver='lbfgs'` (optimización cuasi-Newton)
- `class_weight='balanced'` (compensar desbalance)
- `max_iter=2000` (garantizar convergencia)

### 6.2 Interpretación inicial

**Variables más influyentes (top 3):**
1. **Contract_One year** (β=-1.2): contratos largos reducen fuertemente el churn
2. **InternetService_Fiber optic** (β=0.98): fibra óptica aumenta churn (posible insatisfacción con el servicio)
3. **tenure** (β=-0.72, estandarizado): mayor antigüedad reduce churn

**Signos coherentes:** contratos estables, antigüedad y protecciones reducen churn; cargos altos y servicios premium lo aumentan.

---

## Ejercicio 7 – Umbral de decisión

### 7.1 Umbral por defecto

El umbral 0.5 asume costos iguales para falsos positivos/negativos y clases balanceadas. No es adecuado aquí porque:
- Clases desbalanceadas (73% vs 27%)
- Los costos de error no son simétricos

### 7.2 Falsos positivos y falsos negativos

**En este contexto:**
- **FN (falso negativo):** predecir que un cliente se queda cuando va a cancelar → se pierde al cliente sin acción preventiva (MUY COSTOSO)
- **FP (falso positivo):** predecir churn cuando el cliente se quedará → se gastan recursos de retención innecesariamente (menos costoso)

**Umbral alternativo propuesto: 0.35**

Reduce FN del 45% al 35%, aceptando más FP. Prioriza recall sobre precision, capturando más clientes en riesgo real para campañas de retención preventiva.

---

## Ejercicio 8 – Evaluación del modelo

### 8.1 Matriz de confusión (umbral 0.5)

```
                 Predicho No    Predicho Sí
Real No              1029            104
Real Sí               179            195
```

- **TN=1029:** Clientes retenidos correctamente identificados
- **TP=195:** Churners correctamente identificados (52% del churn real)
- **FN=179:** Churners no detectados (48% perdidos - crítico)
- **FP=104:** Falsas alarmas de retención

### 8.2 Métricas

| Métrica    | Valor  | Interpretación                                    |
|------------|--------|---------------------------------------------------|
| Accuracy   | 0.812  | 81% de predicciones correctas (engañoso por desbalance) |
| Precision  | 0.652  | 65% de alarmas de churn son correctas             |
| Recall     | 0.521  | Solo detectamos 52% de los churners reales       |
| F1-score   | 0.579  | Balance armónico precision-recall                 |
| AUC-ROC    | 0.847  | Buena capacidad de discriminación entre clases   |

**ROC-AUC de 0.847** indica que el modelo tiene 84.7% de probabilidad de rankear correctamente a un churn aleatorio por encima de un no-churn aleatorio.

---

## Ejercicio 9 – Visualización y diagnóstico

**Gráficos generados:**

1. **Distribución de probabilidades:** Muestra separación clara entre clases (churn media=0.42 vs no-churn=0.18), aunque con solapamiento significativo
2. **Curva ROC:** Curvatura pronunciada indica buen poder discriminativo (AUC=0.847)
3. **Precision-Recall:** Útil para clases desbalanceadas; muestra trade-off entre capturar churners (recall) y precisión
4. **Análisis de umbrales:** Visualiza que bajar el umbral a 0.35-0.40 mejora recall del 52% al 68% con caída moderada de precision

Estos gráficos permiten tomar decisiones informadas sobre el umbral óptimo según la estrategia de negocio.

---

## Ejercicio 10 – Conclusiones y pensamiento crítico

**¿El modelo responde al objetivo inicial?**
Sí. Estima probabilidades de churn con AUC=0.847, permitiendo priorizar clientes en riesgo. Sin embargo, con recall=52% en umbral estándar, solo detecta la mitad de los churners.

**Limitaciones:**
- Alta tasa de falsos negativos (48% de churners no detectados)
- Multicolinealidad entre `tenure` y `TotalCharges` puede inflar varianzas
- No captura interacciones complejas (ej: efecto combinado Contract+InternetService)
- Asume relaciones log-lineales que pueden ser simplificaciones

**Mejoras propuestas:**
- Feature engineering: interacciones (Contract × tenure), ratios (TotalCharges/tenure)
- Modelos no lineales: Random Forest, XGBoost para capturar patrones complejos
- Validación cruzada estratificada para estimaciones más robustas
- Análisis de segmentos: entrenar modelos específicos por tipo de contrato

**¿Confiaría en este modelo para decisiones reales?**
Con cautela. Es válido como sistema de priorización (rankear clientes por riesgo), pero no como predictor binario definitivo. Recomendaría:
- Usar umbral ajustado (0.35-0.40) para campañas de retención masiva
- Combinar con análisis cualitativo para clientes de alto valor
- Monitorear performance en producción y re-entrenar periódicamente
- Incorporar feedback de campañas para mejorar el modelo iterativamente



